# PestCLEF 2026: граф знаний + трансформерный NER на GPU

Перед запуском в **Settings** справа включите:
* **Accelerator → GPU T4 x2** (или P100),
* **Internet → On** (нужно для клонирования кода, скачивания EPOP/GeoNames/NCBI и весов модели с Hugging Face).

Ноутбук: (1) обучает NER на `train` и меряет весь пайплайн на `dev`; (2) переобучает NER на `train+dev` и делает предсказания для `test`. Результат — `/kaggle/working/pestclef_outputs.zip`.

In [ ]:
# Модель для NER. Варианты: dmis-lab/biobert-base-cased-v1.2 (по умолчанию),
# dmis-lab/biobert-large-cased-v1.1, microsoft/deberta-v3-base, microsoft/deberta-v3-large, roberta-large
MODEL = "dmis-lab/biobert-base-cased-v1.2"
EPOCHS = 10
BATCH = 4
LR = 5e-5          # для *-large моделей лучше 2e-5
BRANCH = "claude/laughing-hamilton-0c4nn3"


In [ ]:
!nvidia-smi -L
!git clone -q --depth 1 --filter=blob:none --sparse -b $BRANCH https://github.com/neuraCollab/uni.git /kaggle/working/uni
!cd /kaggle/working/uni && git sparse-checkout set competitions/pest-clef-2026
%cd /kaggle/working/uni/competitions/pest-clef-2026
!pip install -q -r requirements.txt rapidfuzz transformers sentencepiece

In [ ]:
# EPOP (тексты train/dev/test + разметка train/dev), GeoNames, NCBI Taxonomy -> data/
!python download_data.py

## 1. NER на train → оценка на dev

In [ ]:
!python train_ner.py --model $MODEL --train train --eval dev --epochs $EPOCHS --batch-size $BATCH --lr $LR

In [ ]:
import os
NAME = MODEL.split('/')[-1]
DEV_MODEL = f"data/models/{NAME}__train"
!python run.py dev                                              # базовая линия: словарный NER
!python run.py dev --ner-model $DEV_MODEL --ner-mode neural
!python run.py dev --ner-model $DEV_MODEL --ner-mode hybrid

## 2. NER на train+dev → предсказания для test
Выберите `NER_MODE` по результатам на dev.

In [ ]:
NER_MODE = "hybrid"
!python train_ner.py --model $MODEL --train train,dev --epochs $EPOCHS --batch-size $BATCH --lr $LR
TEST_MODEL = f"data/models/{NAME}__train+dev"
!python run.py test --ner-model $TEST_MODEL --ner-mode $NER_MODE
!cd outputs && zip -qr /kaggle/working/pestclef_outputs.zip test && ls -la /kaggle/working/pestclef_outputs.zip